In [21]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import torch
torch.set_num_threads(1)

import multiprocessing as mp
import numpy as np

from EffectiveFreqDamp0T import run_simulation

# Define Monte Carlo settings
NUM_SIMULATIONS = 10000
NUM_PROCESSES = 10

if __name__ == "__main__":
    try:
        mp.set_start_method("spawn")
    except RuntimeError:
        pass
    
    length = 50.0
    width  = 2.0
    peddamp = 0.30
    pedBodyF = 3.10
    beamFreq = np.array([2.00])
    beamdamp = np.array([0.005])
    Tocity = 25
    Outcity = 25

    np.random.seed()
    seeds = [np.random.randint(0, 2**31 - 1) for _ in range(NUM_SIMULATIONS)]

    args_list = [(seed, peddamp, pedBodyF, beamFreq, beamdamp, Tocity, Outcity,length, width) for seed in seeds]

    with mp.get_context("spawn").Pool(processes=NUM_PROCESSES) as pool:
        results = pool.map(run_simulation, args_list)

    t_all, eff_damping_all, eff_frequency_all, mass_ratio_all, numped_all = zip(*results)

    t = t_all[0]
    eff_damping_all = np.array(eff_damping_all)
    eff_frequency_all = np.array(eff_frequency_all)
    mass_ratio_all = np.array(mass_ratio_all)
    numped_all = np.array(numped_all)

    print(f"Ran {len(results)} simulations.")
    print("eff_damping_all shape:", eff_damping_all.shape)
    print("eff_frequency_all shape:", eff_frequency_all.shape)
    print("mass_ratio_all shape:", mass_ratio_all.shape)
    print("numped_all shape:", numped_all.shape)

Ran 10000 simulations.
eff_damping_all shape: (10000, 1)
eff_frequency_all shape: (10000, 1)
mass_ratio_all shape: (10000,)
numped_all shape: (10000,)


In [24]:
eff_damping_all = np.array(eff_damping_all)
eff_frequency_all = np.array(eff_frequency_all)
mass_ratio_all = np.array(mass_ratio_all)

# Damping bounds
damp_lower = np.percentile(eff_damping_all, 5)
damp_upper = np.percentile(eff_damping_all, 95)

# Frequency bounds
freq_lower = np.percentile(eff_frequency_all, 5)
freq_upper = np.percentile(eff_frequency_all, 95)

# Mass ratio bounds
mass_lower = np.percentile(mass_ratio_all, 5)
mass_upper = np.percentile(mass_ratio_all, 95)

print(f"Effective damping 5–95% bounds: {damp_lower:.3f} , {damp_upper:.3f}")
print(f"Effective frequency 5–95% bounds: {freq_lower:.3f} , {freq_upper:.3f}")
print(f"Mass ratio 5–95% bounds: {mass_lower:.3f} , {mass_upper:.3f}")

Effective damping 5–95% bounds: 0.018 , 0.024
Effective frequency 5–95% bounds: 1.799 , 1.849
Mass ratio 5–95% bounds: 0.124 , 0.172
